# Task 2 Convolution Neural Network

#### a) 

##### Sources used to guide this task: "https://medium.com/x8-the-ai-community/solving-class-imbalance-problem-in-cnn-9c7a5231c478" and "https://medium.com/@seelcs12/cnns-for-imbalanced-image-classification-with-tensorflow-7284a8c4a2e4"

We start by loading the dataset from CIFAR-10 with all 60000 images. We split them into our training and test data. In addition we also normalize the images as it will be easier for our model to learn from.

In [ ]:
(x_train, y_train), (x_test, y_test) = datasets.cifar10.load_data()

# Normalize pixel values to be between 0 and 1
x_train, x_test = x_train / 255.0, x_test / 255.0

Our task at hand was to create a binary CNN with the CIFAR-10 dataset. After going through most of the data we found out that changing the labels into a binary relation would help greatly into making this CNN fit for our task. In our case we chose that this model should identify frogs, which in the dataset was the label ``[6]``. So we start by going through the dataset to set our new labels for our data.

In [ ]:
num_class = 6

for n in range(len(y_train)):
    if y_train[n] == [num_class]:
        y_train[n] = [1]
    else:
        y_train[n] = [0]

for n in range(len(y_test)):
    if y_test[n] == [num_class]:
        y_test[n] = [1]
    else:
        y_test[n] = [0]

#### Undersampling the data

After some careful feedback and consideration we found that the dataset would be quite imbalanced. As we only have 5000 training images of frogs and 45000 non frog pictures. To make the model understand the frog we decided to undersample our dataset so we have even frog and non frog pictures. We tried to this with our test data as well, but thought that more pictures would mean better results for our model, we come back to this in our evaluation. 

In [ ]:
x_tr_frog = []
y_tr_frog = []
for n in range(len(y_train)):
    if y_train[n] == [1]:
        x_tr_frog.append(x_train[n])
        y_tr_frog.append([1])

for n in range(len(y_train)):
    if len(x_tr_frog) < 10000:
        x_tr_frog.append(x_train[n])
        y_tr_frog.append([0])

x_tr_frog = np.array(x_tr_frog)
y_tr_frog = np.array(y_tr_frog)

#### The model.

Creating the function `make_model`, so we can create two different models with the same model building blocks. Start with 32 units of ``Conv2D`` with the input size set to our pictures size. Went with the solid `relu` activation. Than a `BatchNormalization` which speeds up our CNN to find an optimal solution. Then a `MaxPooling2D` which reduces the shape of our pictures and extract valueable features of the picture. This is our base layer which we use two times in our model before we add the last `Conv2D` and `BatchNormalization`. In the last 'layer' we start adding our `Dense` units to actually train the model. We use the same activation throughout the model, in addition we add a `kernel_regularizer` which helps the model's weights. We also add `Dropout's` units to help with overfitting. Our last `Dense` unit will serve as our output and therefore have the activation `sigmoid` to get the models prediction between 0 and 1. After quite some testing and checking our over/under-fitting graphs and saw quite many spikes we decided to increase our batch size to 128 to better comlpy with the big test set.

In [ ]:
def make_model(metric, x_train, y_train, x_val, y_val):
    with tf.device('/GPU:0'):
        model = models.Sequential()
        model.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)))
        model.add(layers.BatchNormalization())
        model.add(layers.MaxPooling2D((2, 2)))

        model.add(layers.Conv2D(128, (3, 3), activation='relu'))
        model.add(layers.BatchNormalization())
        model.add(layers.MaxPooling2D((2, 2)))

        model.add(layers.Conv2D(256, (3, 3), activation='relu'))
        model.add(layers.BatchNormalization())

        model.add(layers.Flatten())
        model.add(layers.Dense(128, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.005)))
        model.add(layers.Dropout(0.4))
        model.add(layers.Dense(256, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.005)))
        model.add(layers.Dropout(0.4))
        model.add(layers.Dense(1, activation='sigmoid'))

        model.compile(optimizer='adam', loss='binary_crossentropy', metrics=[metric])

        early_stopping = callbacks.EarlyStopping(monitor='val_loss', patience = 5, restore_best_weights = True)

        history = model.fit(
            x_train,
            y_train,
            batch_size = 128,
            epochs = 50,
            validation_data = (x_val, y_val),
            callbacks = [early_stopping]
        )
        return model, history

#### Training the model.

Question to TA: Would it be optimal to compare two different models and explain why the one model is better than the other, or is it better present one model and explain what makes it good.

After some careful consideration and testing with different metrics. We decided to compare two of our models, one with the metric `accuracy`, like our previous models, and one with the `F1-score` metric. The two models use the exact same training set and test set. Both also have an early stopping when our `val_loss` does not improve over five epochs. After training both models we start comparing them in our evaluation. We chose to go for a medium threshold on our `F1Score` since this was quite better than the low and high thresholds.

In [ ]:
acc_model, acc_history = make_model('accuracy', x_tr_frog, y_tr_frog, x_test, y_test)

f1_model, f1_history = make_model(F1Score(threshold=0.5, average='micro'), x_tr_frog, y_tr_frog, x_test, y_test)

#### Evalution graphs

Creating `confusion_matrix` for each model to compare what makes them different. Uses `seaborn` to show the matrices.

In [ ]:
def made_predictions(predictions):
    array = []
    for pred in predictions:
        if max(pred) > 0.7:
            array.append([1])
        else:
            array.append([0])
    return np.array(array)

y_pred_acc = acc_model.predict(x_test)
y_pred_acc = made_predictions(y_pred_acc)
best_cf = confusion_matrix(y_test, y_pred_acc)

y_pred_f1 = f1_model.predict(x_test)
y_pred_f1 = made_predictions(y_pred_f1)
worse_cf = confusion_matrix(y_test, y_pred_f1)

fig, ax = plt.subplots(1,2)
fig.tight_layout()
sns.heatmap(best_cf, annot=True, cmap='Blues', fmt='d', ax=ax[0])
sns.heatmap(worse_cf, annot=True, cmap='Blues', fmt='d', ax=ax[1])
fig.savefig('./matplotlib_data/conf_mat_CNN')
fig.show()

#### Checking underfitting and overfitting

As we mentioned the huge imbalance in our dataset, it is important to chech if our models actually overfit or underfit in our models. We create these two graphs using `seaborn` to check.

In [ ]:
plt.plot(acc_history.history['accuracy'], label='accuracy')
plt.plot(acc_history.history['val_accuracy'], label = 'val_accuracy')
plt.xlabel('Epoch')
plt.ylabel('accuracy')
plt.ylim([0.0, 1])
plt.legend(loc='lower right')
plt.savefig('./matplotlib_data/over_acc_CNN')

test_loss, test_acc = acc_model.evaluate(x_test, y_test, verbose=2)

In [ ]:
plt.plot(f1_history.history['f1_score'], label='f1_score')
plt.plot(f1_history.history['val_f1_score'], label = 'val_f1_score')
plt.xlabel('Epoch')
plt.ylabel('f1_score')
plt.ylim([0.0, 1])
plt.legend(loc='lower right')
plt.savefig('./matplotlib_data/over_f1_CNN')

test_loss, test_acc = f1_model.evaluate(x_test, y_test, verbose=2)

#### Task 2b)

Found these pictures from Google: 

* Image 1: https://www.google.com/url?sa=i&url=https%3A%2F%2Fwww.wcs.org%2Fget-involved%2Fupdates%2Ffascinating-frogs&psig=AOvVaw28_HGqOe7S2sxjPK3bsI5N&ust=1744278715616000&source=images&cd=vfe&opi=89978449&ved=0CBQQjRxqFwoTCKDFld_WyowDFQAAAAAdAAAAABAE

* Image 2: https://www.google.com/url?sa=i&url=https%3A%2F%2Fwww.twinkl.pt%2Fteaching-wiki%2Ffrog&psig=AOvVaw0bdQuBJB9qO9gkRwCWvou-&ust=1744633397275000&source=images&cd=vfe&opi=89978449&ved=0CBQQjRxqFwoTCPCGuIGA1YwDFQAAAAAdAAAAABAE

* Image 3: https://www.google.com/url?sa=i&url=https%3A%2F%2Fwww.nytimes.com%2Fwirecutter%2Freviews%2Fnew-dog-checklist%2F&psig=AOvVaw2ZPUsP1HTJbZhOYM2tqHPx&ust=1744633423068000&source=images&cd=vfe&opi=89978449&ved=0CBQQjRxqFwoTCND_iI6A1YwDFQAAAAAdAAAAABAE

* Image 4: https://www.google.com/url?sa=i&url=https%3A%2F%2Fwww.goodwood.com%2Fgrr%2Ff1%2Fthe-nine-best-f1-cars-of-all-time%2F&psig=AOvVaw0_NSf08TH2_7ChAB8GfZhd&ust=1744633440992000&source=images&cd=vfe&opi=89978449&ved=0CBQQjRxqFwoTCNi76pWA1YwDFQAAAAAdAAAAABAE

Used Tensorflows own `load_img` to convert the pictures to the right size for our models. Then we show how the pictures will look for our models, to see if they have any meaningful similarities that could make the model make a wrong prediction. 

##### Conversion.

After that we convert the images into their array form so we can actually use them in our model. Not much meaningful parameters are used as we just convert and normalize the array as we did earlier. 

In [ ]:
# Showing the pictures we are using to test the models
images = []

img_1 = tf.keras.utils.load_img('./media/frog_6.jpg', target_size=(32, 32))
img_2 = tf.keras.utils.load_img('./media/frog_4.jpg', target_size=(32, 32))
img_3 = tf.keras.utils.load_img('./media/dog_2.jpg', target_size=(32, 32))
img_4 = tf.keras.utils.load_img('./media/f1.jpg', target_size=(32, 32))

images.append(img_1)
images.append(img_2)
images.append(img_3)
images.append(img_4)

fig, ax = plt.subplots(2, 2)
fig.tight_layout()
ax = ax.flatten()

images_for_model = []
num = 1

for img, ax in zip(images, ax):
    img_array = tf.keras.utils.img_to_array(img)
    img_array = np.array(img_array) / 255.0 # Normalize
    images_for_model.append(img_array)
    ax.imshow(img_array)
    ax.set_title(f'Image {num}')
    ax.axis('off')
    num += 1

plt.show()

#### Prediction.

Before we actually predict our images we first have to add a batch so that our model acctually can use our arrays. When we get our prediction back we make it into a percentage so it is more readable. Here we also added a threshhold to decide if we think our models actually predicted right or not. We found that if the model was 70% sure it was a frog it mostly was right. We can see all the predictions below.

In [ ]:
# Actual predictions
print("Hi! I'm Accuracy and I think that: ")
for num in range(len(images_for_model)):
    img = images_for_model[num]
    img = tf.expand_dims(img, 0) # Create a batch
    pred = acc_model.predict(img)
    pred = pred[0][0] * 100
    print(f"    Image {num + 1} is: ", end="")
    if pred > 70:
        print(f'{pred:.2f}% a frog.')
    else:
        print(f'{100 - pred:.2f}% not a frog')

print("\n-----------CHANGING MODEL-----------\n")

print("Hi! I'm F1 and I actually think that: ")
for num in range(len(images_for_model)):
    img = images_for_model[num]
    img = tf.expand_dims(img, 0) # Create a batch
    pred = f1_model.predict(img)
    pred = pred[0][0] * 100
    print(f"    Image {num + 1} is: ", end="")
    if pred > 70:
        print(f'{pred:.2f}% a frog.')
    else:
        print(f'{100 - pred:.2f}% not a frog')# Actual predictions
print("Hi! I'm Accuracy and I think that: ")
for num in range(len(images_for_model)):
    img = images_for_model[num]
    img = tf.expand_dims(img, 0) # Create a batch
    pred = acc_model.predict(img)
    pred = pred[0][0] * 100
    print(f"    Image {num + 1} is: ", end="")
    if pred > 70:
        print(f'{pred:.2f}% a frog.')
    else:
        print(f'{100 - pred:.2f}% not a frog')

print("\n-----------CHANGING MODEL-----------\n")

print("Hi! I'm F1 and I actually think that: ")
for num in range(len(images_for_model)):
    img = images_for_model[num]
    img = tf.expand_dims(img, 0) # Create a batch
    pred = f1_model.predict(img)
    pred = pred[0][0] * 100
    print(f"    Image {num + 1} is: ", end="")
    if pred > 70:
        print(f'{pred:.2f}% a frog.')
    else:
        print(f'{100 - pred:.2f}% not a frog')

### Conclusion of task 2.

To summarize quite short, we have now created two convolutional neural network as a binary classifier of one category. And tested them with pictures from the internet to see if it actually could predict a frog or not. When doing this task is starts quite simple by just building a neural network like we did in task 1. But when actually trying to optimize and find the right parameters, metric, training or test size would be quite tidy to do. When just comparing the two models there is quite significant changes in both. Like when we see at their `confusion matrices`. 

![Conf_matrices](./matplotlib_data/conf_mat_CNN.png)

We easily see a difference in them. Like ``accuracy`` got a better score at finding the non frogs and not classifying them as frogs. But ``F1_Score`` was better at finding the frogs and not defining the frogs as non frogs. This could be more accurate by underfitting the test set, but after some testing with the smaller set the actual percentage different was not significant enough. Which means our models is actually quite optimized at finding the frogs.

Another obstacle we had too overcome was the `val_loss` graphs which we used to check over- and underfitting. As you see here.

![Conf_matrices](./matplotlib_data/over_acc_CNN.png), ![Conf_matrices](./matplotlib_data/over_f1_CNN.png)

Both models showed quite some jumping in `val_loss` which means we have to tweak our batch size even more, or undersample the test set. But as we see in our `accuracy` model it doesn't overfit or underfit as extreme as our `f1_score`. Which means our first model is optimized for the task that was given. When it comes to our second model we see that it underfits quite much. This means that our model is more optimized with a `accuracy` metric than an `F1_Score`. As we could make a more optimized model for the `F1_Score`, by chaning the activations, learning rate or increasing the models capacity, we wanted to compare the same model with two metrics. And by making two different but optimized models would not be our goal.

When it comes to actually testing the models for the images we found online. We can actually see how optimzed our `accuracy` model became. When they are both given the same pictures we see that there are a slight difference, but not as huge as our graphs. Like with `Image 1` we see that `Accuracy` was much more determined that it was a frog than our `F1_Score` which was right at our threshold with 1,34%. With `Image 2` they were both certain that it was a frog. And with `Image 3` and `Image 4` both models thought that it wasn't a frog, which is correct, but only that `Accuracy` was much more decisive than `F1_Score`. 

As we conclude our task we have learnt much about the convolutional neural network, like how our model was more optimized with an `accuracy` metric than `F1_Score`. How the networks actually analyze pictures and how to test own pictures with our own model.